In [ ]:
# Structure features with top-2.5% structure perturbation

from nupack import *
import RNA
import math
import itertools

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def get_segments_interrupted_by_plus(input_string: str) -> list:
    # Split the string by the "+" character
    segments = input_string.split("+")
    
    # Remove empty segments if any (e.g., if the string starts or ends with "+")
    segments = [segment for segment in segments if segment]
    
    return segments

def encode_structure_symbol(symbol):
    if symbol == '.':
        return [1, 0, 0]
    elif symbol == '(':
        return [0, 1, 0]
    elif symbol == ')':
        return [0, 0, 1]
    else:
        return [0, 0, 0]

def get_structure_string(subopt_result, structure_rank=0):
    """
    structure_rank = 0: highest-probability / lowest-energy structure
    structure_rank = 1: second-highest-probability structure
    """
    if len(subopt_result) > structure_rank:
        return str(subopt_result[structure_rank].structure)
    else:
        return str(subopt_result[0].structure)

def calculate_probability_gap(subopt_result, strands, model):
    if len(subopt_result) < 2:
        return 0.0

    prob_top = structure_probability(
        strands=strands,
        structure=subopt_result[0].structure,
        model=model
    )

    prob_second = structure_probability(
        strands=strands,
        structure=subopt_result[1].structure,
        model=model
    )

    return prob_top - prob_second

file_paths = [
    'Feature_CNN2_0025.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('off_target_full_guide_spacer_sequences_EIF3B_filtered_sampled_1.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('off_target_noPAM_target_sequences_EIF3B_filtered_sampled_1.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('off_target_noPAM_target_sequences_EIF3B_filtered_sampled_1.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

file4 = open('off_target_matched_positions_EIF3B_sampled_1.txt', 'r')    
lines = file4.readlines()
start_position_array = [line.strip() for line in lines]

file5 = open('NM_001362792.2.txt', 'r')
complete_target_part1 = file5.readlines()
complete_target_part1 = complete_target_part1[0]

subopt_structures_target_bh_part1 = subopt(strands=complete_target_part1, energy_gap=0.1, model=my_model_RNA)

# ============================================================
# First pass:
# rank samples by top-vs-second structure probability gap
# ============================================================

guide_gap_list = []
target_gap_list = []

all_guide_bh_subopts = []
all_hybrid_ah_subopts = []

target_gap_part1 = calculate_probability_gap(
    subopt_structures_target_bh_part1,
    strands=complete_target_part1,
    model=my_model_RNA
)

for i in range(len(guide_array)):

    guide = guide_array[i]
    target_truncated = truncated_target_array[i]

    subopt_structures_guide_bh = subopt(
        strands=guide,
        energy_gap=3,
        model=my_model_RNA
    )

    subopt_structures_hybrid_ah = subopt(
        strands=[guide, target_truncated],
        energy_gap=3,
        model=my_model_RNA
    )

    all_guide_bh_subopts.append(subopt_structures_guide_bh)
    all_hybrid_ah_subopts.append(subopt_structures_hybrid_ah)

    guide_gap = calculate_probability_gap(
        subopt_structures_guide_bh,
        strands=guide,
        model=my_model_RNA
    )

    if i < 10000:
        target_gap = target_gap_part1

    guide_gap_list.append((i, guide_gap))
    target_gap_list.append((i, target_gap))

top_percent_count = max(1, math.ceil(len(guide_array) * 0.025))

guide_switch_indices = set(
    idx for idx, gap in sorted(guide_gap_list, key=lambda x: x[1], reverse=True)[:top_percent_count]
)

target_switch_indices = set(
    idx for idx, gap in sorted(target_gap_list, key=lambda x: x[1], reverse=True)[:top_percent_count]
)

print("Top 1% guide_bh samples switched to second-highest-probability structure:")
print(sorted(guide_switch_indices))

print("Top 1% target_bh samples switched to second-highest-probability structure:")
print(sorted(target_switch_indices))

# ============================================================
# Second pass:
# encode structures
# ============================================================

for i in range (0, len(guide_array)):

    # Initialize struct_array
    struct_array = []
    struct_array_unit = []

    subopt_structures_guide_bh = all_guide_bh_subopts[i]
    subopt_structures_hybrid_ah = all_hybrid_ah_subopts[i]

    if i in guide_switch_indices:
        guide_bh_structure = get_structure_string(subopt_structures_guide_bh, structure_rank=1)
    else:
        guide_bh_structure = get_structure_string(subopt_structures_guide_bh, structure_rank=0)

    if i < 10000:
        if i in target_switch_indices:
            target_bh_structure = get_structure_string(subopt_structures_target_bh_part1, structure_rank=1)
        else:
            target_bh_structure = get_structure_string(subopt_structures_target_bh_part1, structure_rank=0)

    hybrid_ah_structure = get_structure_string(subopt_structures_hybrid_ah, structure_rank=0)

    if i < 10000:
        
        # Guide structure before hybridization
        for j in range (0, 53):
            
            try:
                if guide_bh_structure[j] == '.':
                    to_be_added = ([1, 0, 0])
                elif guide_bh_structure[j] == '(':
                    to_be_added = ([0, 1, 0])
                elif guide_bh_structure[j] == ')':
                    to_be_added = ([0, 0, 1])
                else:
                    to_be_added = ([0, 0, 0])
                    
            except IndexError:
                to_be_added = ([0, 0, 0])

            struct_array_unit.append(to_be_added)
            
        # Target structure before hybridization 
        for j in range (int(start_position_array[i])-1, int(start_position_array[i]) + 22):
            
            try:
                if target_bh_structure[j] == '.':
                    to_be_added = ([1, 0, 0])
                elif target_bh_structure[j] == '(':
                    to_be_added = ([0, 1, 0])
                elif target_bh_structure[j] == ')':
                    to_be_added = ([0, 0, 1])
                else:
                    to_be_added = ([0, 0, 0])
                    
            except IndexError:
                to_be_added = ([0, 0, 0])
    
            struct_array_unit.append(to_be_added)

         
        # Hybrid structure after hybridization 
        hybrid_ah_1 = get_segments_interrupted_by_plus(hybrid_ah_structure)[0]
        hybrid_ah_2 = get_segments_interrupted_by_plus(hybrid_ah_structure)[1]
    
        for j in range (0, 53):
            
            try:
                if hybrid_ah_1[j] == '.':
                    to_be_added = ([1, 0, 0])
                elif hybrid_ah_1[j] == '(':
                    to_be_added = ([0, 1, 0])
                elif hybrid_ah_1[j] == ')':
                    to_be_added = ([0, 0, 1])
                else:
                    to_be_added = ([0, 0, 0])
                
            except IndexError:
                 to_be_added = ([0, 0, 0])
            
            struct_array_unit.append(to_be_added)

     
        for j in range (0, 23):
            
            try:
                if hybrid_ah_2[j] == '.':
                    to_be_added = ([1, 0, 0])
                elif hybrid_ah_2[j] == '(':
                    to_be_added = ([0, 1, 0])
                elif hybrid_ah_2[j] == ')':
                    to_be_added = ([0, 0, 1])
                else:
                    to_be_added = ([0, 0, 0])
                
            except IndexError:
                 to_be_added = ([0, 0, 0])
            
            struct_array_unit.append(to_be_added)
            
    struct_array_to_append = [struct_array_unit]
    struct_array.append(struct_array_to_append)

    struct_array = list(itertools.chain.from_iterable(struct_array))
    
    # Open a file in write mode
    with open('Feature_CNN2_0025.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')

In [ ]:
# Energy features
# Gaussian noise: Mean = 0, STD = 0.05 (after normalization)

from nupack import *
import RNA
import math
import itertools
import numpy as np

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))

def RNA_to_DNA(RNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'U': 'T'}
    return ''.join(match.get(base, base) for base in (RNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def find_substring_positions(substring, long_string):

    positions = []
    start = 0
    while True:
        index = long_string.find(substring, start)
        if index == -1:  # No more occurrences found.
            break
        positions.append(index)
        start = index + 1  # Move past the last found index.
    return positions

def normalize_guide(value):
    normalized_value = (value - (-13)) / (0 - (-13))
    return normalized_value

def normalize_target_bh(value):
    normalized_value = (value - (-47)) / (-1 - (-47))
    return normalized_value

def normalize_target(value):
    normalized_value = (value - (-71)) / (-28 - (-71))
    return normalized_value
    

def normalize_guide_conse(value):
    normalized_value = (value - (-34)) / (2 - (-34))
    return normalized_value

def normalize_target_bh_conse(value):
    normalized_value = (value - (-39)) / (3 - (-39))
    return normalized_value
    
def normalize_target_conse(value):
    normalized_value = (value - (-60)) / (-2 - (-60))
    return normalized_value
    

def normalize_guide_conse_unpaired(value):
    normalized_value = (value - (-56)) / (2 - (-56))
    return normalized_value

def normalize_target_bh_conse_unpaired(value):
    normalized_value = (value - (-37)) / (3 - (-37))
    return normalized_value

def normalize_target_conse_unpaired(value):
    normalized_value = (value - (-12)) / (3 - (-12))
    return normalized_value
    

def normalize_guide_overhang(value):
    normalized_value = (value - (-56)) / (3 - (-56))
    return normalized_value

def normalize_target_bh_overhang(value):
    normalized_value = (value - (-35)) / (3 - (-35))
    return normalized_value

def normalize_target_overhang(value):
    normalized_value = (value - (-12)) / (3 - (-12))
    return normalized_value

def normalize_guide_paired(value):
    normalized_value = (value - (-32)) / (3 - (-32))
    return normalized_value

def normalize_target_bh_paired(value):
    normalized_value = (value - (-38)) / (3 - (-38))
    return normalized_value

def normalize_target_paired(value):
    normalized_value = (value - (-60)) / (3 - (-60))
    return normalized_value

def normalize_seed(value):
    normalized_value = (value - (-21)) / (-3 - (-21))
    return normalized_value

def normalize_middle(value):
    normalized_value = (value - (-21)) / (-3 - (-21))
    return normalized_value

def normalize_distal(value):
    normalized_value = (value - (-17)) / (-2 - (-17))
    return normalized_value

def find_max_base_pairs(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '()':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def find_max_unpaired(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '.':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def detect_5_overhang(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_overhang(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def detect_5_paired(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_paired(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def is_all_parens(s):
    for char in s:
        if char not in ("(", ")"):
            return False
    return True

# Record positions of base pairs with the order of base positions in "left_paren" corresponding to the base positions in "right_paren"
def find_pair(structure):
    left_paren = []
    right_paren = []
    left_paren_pre = []

    for i in range (0, len(structure)):
        if structure[i] == '(':
            left_paren_pre.append(i)
        if structure[i] == ')':
            right_paren.append(i)
            left_paren.append(left_paren_pre[-1])
            left_paren_pre = left_paren_pre[:-1]

    return left_paren, right_paren

def pairing_state_correction(structure, left_paren, right_paren, start_position, end_position):
    structure_list = list(structure)
    for i in range (0, len(left_paren)):
        if left_paren[i] >= start_position and left_paren[i] <= end_position:
            structure_list[left_paren[i]] = '.'
            structure_list[right_paren[i]] = '.'
        if right_paren[i] >= start_position and right_paren[i] <= end_position:
            structure_list[left_paren[i]] = '.'
            structure_list[right_paren[i]] = '.'
        structure = ''.join(structure_list)
    return structure
    

file_paths = [
    'Feature_MLP_0025.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('off_target_full_guide_spacer_sequences_EIF3B_filtered_sampled_1.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('off_target_noPAM_target_sequences_EIF3B_filtered_sampled_1.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('off_target_noPAM_target_sequences_EIF3B_filtered_sampled_1.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

file4 = open('off_target_matched_positions_EIF3B_sampled_1.txt', 'r')    
lines = file4.readlines()
start_position_array = [line.strip() for line in lines]

file5 = open('NM_001362792.2.txt', 'r')
complete_target_part1 = file5.readlines()
complete_target_part1 = complete_target_part1[0]

min_guide_energy = np.inf
max_guide_energy = -np.inf
min_target_bh_energy = np.inf
max_target_bh_energy = -np.inf
min_target_energy = np.inf
max_target_energy = -np.inf

min_guide_energy_conse = np.inf
max_guide_energy_conse = -np.inf
min_target_bh_energy_conse = np.inf
max_target_bh_energy_conse = -np.inf
min_target_energy_conse = np.inf
max_target_energy_conse = -np.inf

min_guide_energy_conse_unpaired = np.inf
max_guide_energy_conse_unpaired = -np.inf
min_target_bh_energy_conse_unpaired = np.inf
max_target_bh_energy_conse_unpaired = -np.inf
min_target_energy_conse_unpaired = np.inf
max_target_energy_conse_unpaired = -np.inf

min_guide_energy_5_overhang = np.inf
max_guide_energy_5_overhang = -np.inf
min_target_bh_energy_5_overhang = np.inf
max_target_bh_energy_5_overhang = -np.inf
min_target_energy_5_overhang = np.inf
max_target_energy_5_overhang = -np.inf

min_guide_energy_3_overhang = np.inf
max_guide_energy_3_overhang = -np.inf
min_target_bh_energy_3_overhang = np.inf
max_target_bh_energy_3_overhang = -np.inf
min_target_energy_3_overhang = np.inf
max_target_energy_3_overhang = -np.inf

min_guide_energy_5_paired = np.inf
max_guide_energy_5_paired = -np.inf
min_target_bh_energy_5_paired = np.inf
max_target_bh_energy_5_paired = -np.inf
min_target_energy_5_paired = np.inf
max_target_energy_5_paired = -np.inf

min_guide_energy_3_paired = np.inf
max_guide_energy_3_paired = -np.inf
min_target_bh_energy_3_paired = np.inf
max_target_bh_energy_3_paired = -np.inf
min_target_energy_3_paired = np.inf
max_target_energy_3_paired = -np.inf

max_seed_energy = -np.inf
min_seed_energy = np.inf

max_middle_energy = -np.inf
min_middle_energy = np.inf

max_distal_energy = -np.inf
min_distal_energy = np.inf

 # Initialize energy_array
energy_array = []

partition_function_target_bh_part1 = pfunc(strands=complete_target_part1, model=my_model_RNA)
ensemble_energy_target_bh_part1 = partition_function_target_bh_part1[1]
subopt_structures_target_bh_part1 = subopt(strands=complete_target_part1, energy_gap=0.000001, model=my_model_RNA)

# ===== Gaussian-noise settings =====

NOISE_STD = 0.05      # 5% standard deviation
NOISE_PERCENT = 0.025  # perturb 2.5% of samples for each feature independently

num_samples = len(guide_array)
num_features = 24

# Random sample indices for each feature
feature_noise_indices = []

for feature_idx in range(num_features):
    indices = np.random.choice(
        num_samples,
        size=max(1, int(np.ceil(num_samples * NOISE_PERCENT))),
        replace=False
    )
    feature_noise_indices.append(set(indices))

for i in range (0, len(guide_array)):
    
    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]
    start_position = int(start_position_array[i])

    # Initialize energy_array
    energy_array_guide = []
    energy_array_target = []

    # Compute ensemble energy
    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    
    partition_function_target = pfunc(strands=[guide, target_truncated], model=my_model_RNA)
    ensemble_energy_target = partition_function_target[1]
    
    # Compute suboptimal structures and energy
    subopt_structures_guide = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_target = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_RNA)

    if i < 10000:
        energy_array_unit = []
        
        # Get paired base positions in ssRNA target before hybridization
        left_paren_target_bh, right_paren_target_bh = find_pair(target_bh_structure)
        corrected_target_structure_bh = pairing_state_correction(target_bh_structure, left_paren_target_bh, right_paren_target_bh, start_position, start_position+22)   

        len_scaffold = 30
        len_spacer = 23
    
        # Calculate/compare subopt_structures_guide[0].energy and subopt_structures_target[0].energy
        if subopt_structures_guide[0].energy + 11.88 > max_guide_energy:
            max_guide_energy = subopt_structures_guide[0].energy + 11.88
        if subopt_structures_guide[0].energy + 11.88 < min_guide_energy:
            min_guide_energy = subopt_structures_guide[0].energy + 11.88
    
        standard_energy_target_bh = structure_energy(strands=complete_target_part1, structure=corrected_target_structure_bh, model=my_model_RNA)
        if subopt_structures_target_bh_part1[0].energy - standard_energy_target_bh > max_target_bh_energy:
            max_target_bh_energy = subopt_structures_target_bh_part1[0].energy - standard_energy_target_bh
        if subopt_structures_target_bh_part1[0].energy - standard_energy_target_bh < min_target_bh_energy:
            min_target_bh_energy = subopt_structures_target_bh_part1[0].energy - standard_energy_target_bh
        
        if subopt_structures_target[0].energy > max_target_energy:
            max_target_energy = subopt_structures_target[0].energy
        if subopt_structures_target[0].energy < min_target_energy:
            min_target_energy = subopt_structures_target[0].energy

        # Calculate/compare ensemble_energy_max_paired_guide and ensemble_energy_max_paired_target
        if str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer] == '.' * len_spacer:
            ensemble_energy_max_paired_guide = 0
        else:
            max_start_index_guide, max_length_guide = find_max_base_pairs(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_paired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
            partition_function_max_paired_guide = pfunc(strands=[max_paired_seq_guide, RNA_reverse_complement(max_paired_seq_guide)], model=my_model_RNA)
            ensemble_energy_max_paired_guide = partition_function_max_paired_guide[1]

        if target_bh_structure[start_position:start_position+23] == '.' * len_spacer:
            ensemble_energy_max_paired_target_bh = 0
        else:
            max_start_index_target_bh, max_length_target_bh = find_max_base_pairs(target_bh_structure[start_position:start_position+23])
            max_paired_seq_target_bh = target_truncated[max_start_index_target_bh:max_start_index_target_bh+max_length_target_bh]
            partition_function_max_paired_target_bh = pfunc(strands=[max_paired_seq_target_bh, RNA_reverse_complement(max_paired_seq_target_bh)], model=my_model_RNA)
            ensemble_energy_max_paired_target_bh = partition_function_max_paired_target_bh[1]

        if str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer] == '.' * len_spacer:
            ensemble_energy_max_paired_target = 0
        else:
            max_start_index_target, max_length_target = find_max_base_pairs(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_paired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
            RNA_max_paired_seq_target = max_paired_seq_target
            partition_function_max_paired_target = pfunc(strands=[RNA_max_paired_seq_target, RNA_reverse_complement(RNA_max_paired_seq_target)], model=my_model_RNA)
            ensemble_energy_max_paired_target = partition_function_max_paired_target[1]

        if ensemble_energy_max_paired_guide > max_guide_energy_conse:
            max_guide_energy_conse = ensemble_energy_max_paired_guide
        if ensemble_energy_max_paired_guide < min_guide_energy_conse:
            min_guide_energy_conse = ensemble_energy_max_paired_guide
    
        if ensemble_energy_max_paired_target_bh > max_target_bh_energy_conse:
            max_target_bh_energy_conse = ensemble_energy_max_paired_target_bh
        if ensemble_energy_max_paired_target_bh < min_target_bh_energy_conse:
            min_target_bh_energy_conse = ensemble_energy_max_paired_target_bh
    
        if ensemble_energy_max_paired_target > max_target_energy_conse:
            max_target_energy_conse = ensemble_energy_max_paired_target
        if ensemble_energy_max_paired_target < min_target_energy_conse:
            min_target_energy_conse = ensemble_energy_max_paired_target

    
        # Calculate/compare ensemble_energy_max_unpaired_guide and ensemble_energy_max_unpaired_target
        if is_all_parens(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer]) == True:
            ensemble_energy_max_unpaired_guide = 0
        else:
            max_start_index_guide, max_length_guide = find_max_unpaired(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_unpaired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
            partition_function_max_unpaired_guide = pfunc(strands=[max_unpaired_seq_guide, RNA_reverse_complement(max_unpaired_seq_guide)], model=my_model_RNA)
            ensemble_energy_max_unpaired_guide = partition_function_max_unpaired_guide[1]
    
        if is_all_parens(target_bh_structure[start_position:start_position+23]) == True:
            ensemble_energy_max_unpaired_target_bh = 0
        else:
            max_start_index_target_bh, max_length_target_bh = find_max_unpaired(target_bh_structure[start_position:start_position+23])
            max_unpaired_seq_target_bh = target_truncated[max_start_index_target_bh:max_start_index_target_bh+max_length_target_bh]
            partition_function_max_unpaired_target_bh = pfunc(strands=[max_unpaired_seq_target_bh, RNA_reverse_complement(max_unpaired_seq_target_bh)], model=my_model_RNA)
            ensemble_energy_max_unpaired_target_bh = partition_function_max_unpaired_target_bh[1]
    
        if is_all_parens(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer]) == True:
            ensemble_energy_max_unpaired_target = 0
        else:
            max_start_index_target, max_length_target = find_max_unpaired(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_unpaired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
            RNA_max_unpaired_seq_target = max_unpaired_seq_target
            partition_function_max_unpaired_target = pfunc(strands=[RNA_max_unpaired_seq_target, RNA_reverse_complement(RNA_max_unpaired_seq_target)], model=my_model_RNA)
            ensemble_energy_max_unpaired_target = partition_function_max_unpaired_target[1]

        if ensemble_energy_max_unpaired_guide > max_guide_energy_conse_unpaired:
            max_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide
        if ensemble_energy_max_unpaired_guide < min_guide_energy_conse_unpaired:
            min_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide
    
        if ensemble_energy_max_unpaired_target_bh > max_target_bh_energy_conse_unpaired:
            max_target_bh_energy_conse_unpaired = ensemble_energy_max_unpaired_target_bh
        if ensemble_energy_max_unpaired_target_bh < min_target_bh_energy_conse_unpaired:
            min_target_bh_energy_conse_unpaired = ensemble_energy_max_unpaired_target_bh
    
        if ensemble_energy_max_unpaired_target > max_target_energy_conse_unpaired:
            max_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target
        if ensemble_energy_max_unpaired_target < min_target_energy_conse_unpaired:
            min_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target

        # Calculate/compare ensemble_energy_5_overhang_guide and ensemble_energy_5_overhang_target
        if str(subopt_structures_guide[0].structure)[len_scaffold] != '.':
            ensemble_energy_5_overhang_guide = 0
        else:
            max_start_index_guide, max_length_guide = detect_5_overhang(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_5_overhang_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
            partition_function_5_overhang_guide = pfunc(strands=[max_5_overhang_seq_guide, RNA_reverse_complement(max_5_overhang_seq_guide)], model=my_model_RNA)
            ensemble_energy_5_overhang_guide = partition_function_5_overhang_guide[1]
    
        if target_bh_structure[start_position] != '.':
            ensemble_energy_5_overhang_target_bh = 0
        else:
            max_start_index_target_bh, max_length_target_bh = detect_5_overhang(target_bh_structure[start_position:start_position+23])
            max_5_overhang_seq_target_bh = target_truncated[max_start_index_target_bh:max_start_index_target_bh+max_length_target_bh]
            partition_function_5_overhang_target_bh = pfunc(strands=[max_5_overhang_seq_target_bh, RNA_reverse_complement(max_5_overhang_seq_target_bh)], model=my_model_RNA)
            ensemble_energy_5_overhang_target_bh = partition_function_5_overhang_target_bh[1]
    
        if str(subopt_structures_target[0].structure)[len_scaffold] != '.':
            ensemble_energy_5_overhang_target = 0
        else:
            max_start_index_target, max_length_target = detect_5_overhang(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])       
            max_5_overhang_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
            RNA_5_overhang_seq_target = max_5_overhang_seq_target
            partition_function_5_overhang_target = pfunc(strands=[RNA_5_overhang_seq_target, RNA_reverse_complement(RNA_5_overhang_seq_target)], model=my_model_RNA)
            ensemble_energy_5_overhang_target = partition_function_5_overhang_target[1]

        if ensemble_energy_5_overhang_guide > max_guide_energy_5_overhang:
            max_guide_energy_5_overhang = ensemble_energy_5_overhang_guide
        if ensemble_energy_5_overhang_guide < min_guide_energy_5_overhang:
            min_guide_energy_5_overhang = ensemble_energy_5_overhang_guide
    
        if ensemble_energy_5_overhang_target_bh > max_target_bh_energy_5_overhang:
            max_target_bh_energy_5_overhang = ensemble_energy_5_overhang_target_bh
        if ensemble_energy_5_overhang_target_bh < min_target_bh_energy_5_overhang:
            min_target_bh_energy_5_overhang = ensemble_energy_5_overhang_target_bh
    
        if ensemble_energy_5_overhang_target > max_target_energy_5_overhang:
            max_target_energy_5_overhang = ensemble_energy_5_overhang_target
        if ensemble_energy_5_overhang_target < min_target_energy_5_overhang:
            min_target_energy_5_overhang = ensemble_energy_5_overhang_target
            

        # Calculate/compare ensemble_energy_3_overhang_guide and ensemble_energy_3_overhang_target
        if str(subopt_structures_guide[0].structure)[len_scaffold+len_spacer-1] != '.':
            ensemble_energy_3_overhang_guide = 0
        else:
            max_start_index_guide, max_length_guide = detect_3_overhang(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_3_overhang_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
            partition_function_3_overhang_guide = pfunc(strands=[max_3_overhang_seq_guide, RNA_reverse_complement(max_3_overhang_seq_guide)], model=my_model_RNA)
            ensemble_energy_3_overhang_guide = partition_function_3_overhang_guide[1]
    
        if target_bh_structure[start_position+22] != '.':
            ensemble_energy_3_overhang_target_bh = 0
        else:
            max_start_index_target_bh, max_length_target_bh = detect_3_overhang(target_bh_structure[start_position:start_position+23])
            max_3_overhang_seq_target_bh = target_truncated[max_start_index_target_bh:max_start_index_target_bh+max_length_target_bh]
            partition_function_3_overhang_target_bh = pfunc(strands=[max_3_overhang_seq_target_bh, RNA_reverse_complement(max_3_overhang_seq_target_bh)], model=my_model_RNA)
            ensemble_energy_3_overhang_target_bh = partition_function_3_overhang_target_bh[1]

        if str(subopt_structures_target[0].structure)[len_scaffold+len_spacer-1] != '.':
            ensemble_energy_3_overhang_target = 0
        else:
            max_start_index_target, max_length_target = detect_3_overhang(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_3_overhang_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
            RNA_3_overhang_seq_target = max_3_overhang_seq_target
            partition_function_3_overhang_target = pfunc(strands=[RNA_3_overhang_seq_target, RNA_reverse_complement(RNA_3_overhang_seq_target)], model=my_model_RNA)
            ensemble_energy_3_overhang_target = partition_function_3_overhang_target[1]
    
        if ensemble_energy_3_overhang_guide > max_guide_energy_3_overhang:
            max_guide_energy_3_overhang = ensemble_energy_3_overhang_guide
        if ensemble_energy_3_overhang_guide < min_guide_energy_3_overhang:
            min_guide_energy_3_overhang = ensemble_energy_3_overhang_guide
    
        if ensemble_energy_3_overhang_target_bh > max_target_bh_energy_3_overhang:
            max_target_bh_energy_3_overhang = ensemble_energy_3_overhang_target_bh
        if ensemble_energy_3_overhang_target_bh < min_target_bh_energy_3_overhang:
            min_target_bh_energy_3_overhang = ensemble_energy_3_overhang_target_bh
    
        if ensemble_energy_3_overhang_target > max_target_energy_3_overhang:
            max_target_energy_3_overhang = ensemble_energy_3_overhang_target
        if ensemble_energy_3_overhang_target < min_target_energy_3_overhang:
            min_target_energy_3_overhang = ensemble_energy_3_overhang_target
            

        # Calculate/compare ensemble_energy_5_paired_guide and ensemble_energy_5_paired_target
        if str(subopt_structures_guide[0].structure)[len_scaffold] == '.':
            ensemble_energy_5_paired_guide = 0
        else:
            max_start_index_guide, max_length_guide = detect_5_paired(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_5_paired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
            partition_function_5_paired_guide = pfunc(strands=[max_5_paired_seq_guide, RNA_reverse_complement(max_5_paired_seq_guide)], model=my_model_RNA)
            ensemble_energy_5_paired_guide = partition_function_5_paired_guide[1]
    
        if target_bh_structure[start_position] == '.':
            ensemble_energy_5_paired_target_bh = 0
        else:
            max_start_index_target_bh, max_length_target_bh = detect_5_paired(target_bh_structure[start_position:start_position+23])
            max_5_paired_seq_target_bh = target_truncated[max_start_index_target_bh:max_start_index_target_bh+max_length_target_bh]
            partition_function_5_paired_target_bh = pfunc(strands=[max_5_paired_seq_target_bh, RNA_reverse_complement(max_5_paired_seq_target_bh)], model=my_model_RNA)
            ensemble_energy_5_paired_target_bh = partition_function_5_paired_target_bh[1]
    
        if str(subopt_structures_target[0].structure)[len_scaffold] == '.':
            ensemble_energy_5_paired_target = 0
        else:
            max_start_index_target, max_length_target = detect_5_paired(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_5_paired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
            RNA_5_paired_seq_target = max_5_paired_seq_target
            partition_function_5_paired_target = pfunc(strands=[RNA_5_paired_seq_target, RNA_reverse_complement(RNA_5_paired_seq_target)], model=my_model_RNA)
            ensemble_energy_5_paired_target = partition_function_5_paired_target[1]
    
        if ensemble_energy_5_paired_guide > max_guide_energy_5_paired:
            max_guide_energy_5_paired = ensemble_energy_5_paired_guide
        if ensemble_energy_5_paired_guide < min_guide_energy_5_paired:
            min_guide_energy_5_paired = ensemble_energy_5_paired_guide

        if ensemble_energy_5_paired_target_bh > max_target_bh_energy_5_paired:
            max_target_bh_energy_5_paired = ensemble_energy_5_paired_target_bh
        if ensemble_energy_5_paired_target_bh < min_target_bh_energy_5_paired:
            min_target_bh_energy_5_paired = ensemble_energy_5_paired_target_bh
    
        if ensemble_energy_5_paired_target > max_target_energy_5_paired:
            max_target_energy_5_paired = ensemble_energy_5_paired_target
        if ensemble_energy_5_paired_target < min_target_energy_5_paired:
            min_target_energy_5_paired = ensemble_energy_5_paired_target
    
        # Calculate/compare ensemble_energy_3_paired_guide and ensemble_energy_3_paired_target
        if str(subopt_structures_guide[0].structure)[len_scaffold+len_spacer-1] == '.':
            ensemble_energy_3_paired_guide = 0
        else:
            max_start_index_guide, max_length_guide = detect_3_paired(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_3_paired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
            partition_function_3_paired_guide = pfunc(strands=[max_3_paired_seq_guide, RNA_reverse_complement(max_3_paired_seq_guide)], model=my_model_RNA)
            ensemble_energy_3_paired_guide = partition_function_3_paired_guide[1]
    
        if target_bh_structure[start_position+22] == '.':
            ensemble_energy_3_paired_target_bh = 0
        else:
            max_start_index_target_bh, max_length_target_bh = detect_3_paired(target_bh_structure[start_position:start_position+23])
            max_3_paired_seq_target_bh = target_truncated[max_start_index_target_bh:max_start_index_target_bh+max_length_target_bh]
            partition_function_3_paired_target_bh = pfunc(strands=[max_3_paired_seq_target_bh, RNA_reverse_complement(max_3_paired_seq_target_bh)], model=my_model_RNA)
            ensemble_energy_3_paired_target_bh = partition_function_3_paired_target_bh[1]

        if str(subopt_structures_target[0].structure)[len_scaffold+len_spacer-1] == '.':
            ensemble_energy_3_paired_target = 0
        else:
            max_start_index_target, max_length_target = detect_3_paired(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
            max_3_paired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
            RNA_3_paired_seq_target = max_3_paired_seq_target
            partition_function_3_paired_target = pfunc(strands=[RNA_3_paired_seq_target, RNA_reverse_complement(RNA_3_paired_seq_target)], model=my_model_RNA)
            ensemble_energy_3_paired_target = partition_function_3_paired_target[1]
    
        if ensemble_energy_3_paired_guide > max_guide_energy_3_paired:
            max_guide_energy_3_paired = ensemble_energy_3_paired_guide
        if ensemble_energy_3_paired_guide < min_guide_energy_3_paired:
            min_guide_energy_3_paired = ensemble_energy_3_paired_guide
    
        if ensemble_energy_3_paired_target_bh > max_target_bh_energy_3_paired:
            max_target_bh_energy_3_paired = ensemble_energy_3_paired_target_bh
        if ensemble_energy_3_paired_target_bh < min_target_bh_energy_3_paired:
            min_target_bh_energy_3_paired = ensemble_energy_3_paired_target_bh
    
        if ensemble_energy_3_paired_target > max_target_energy_3_paired:
            max_target_energy_3_paired = ensemble_energy_3_paired_target
        if ensemble_energy_3_paired_target < min_target_energy_3_paired:
            min_target_energy_3_paired = ensemble_energy_3_paired_target

        # Calculate/compare target seed region free energy 
        seed_region = target_truncated[15:23]
        subopt_structures_seed = subopt(strands=[seed_region, RNA_reverse_complement(seed_region)], energy_gap=3, model=my_model_RNA)
        seed_energy = subopt_structures_seed[0].energy
    
        if seed_energy > max_seed_energy:
            max_seed_energy = seed_energy
        if seed_energy < min_seed_energy:
            min_seed_energy = seed_energy
    
    
        # Calculate/compare target middle region free energy 
        middle_region = target_truncated[7:15]
        subopt_structures_middle = subopt(strands=[middle_region, RNA_reverse_complement(middle_region)], energy_gap=3, model=my_model_RNA)
        middle_energy = subopt_structures_middle[0].energy
    
        if middle_energy > max_middle_energy:
            max_middle_energy = middle_energy
        if middle_energy < min_middle_energy:
            min_middle_energy = middle_energy

        # Calculate/compare target distal region free energy 
        distal_region = target_truncated[0:7]
        subopt_structures_distal = subopt(strands=[distal_region, RNA_reverse_complement(distal_region)], energy_gap=3, model=my_model_RNA)
        distal_energy = subopt_structures_distal[0].energy
    
        if distal_energy > max_distal_energy:
            max_distal_energy = distal_energy
        if distal_energy < min_distal_energy:
            min_distal_energy = distal_energy
            

        to_be_added = [normalize_guide(subopt_structures_guide[0].energy + 11.88)]
        to_be_added.append(normalize_guide_conse(ensemble_energy_max_paired_guide))
        to_be_added.append(normalize_guide_conse_unpaired(ensemble_energy_max_unpaired_guide))
        to_be_added.append(normalize_guide_overhang(ensemble_energy_5_overhang_guide))
        to_be_added.append(normalize_guide_overhang(ensemble_energy_3_overhang_guide))
        to_be_added.append(normalize_guide_paired(ensemble_energy_5_paired_guide))
        to_be_added.append(normalize_guide_paired(ensemble_energy_3_paired_guide))
    
        to_be_added.append(normalize_target_bh(subopt_structures_target_bh_part1[0].energy - standard_energy_target_bh))
        to_be_added.append(normalize_target_bh_conse(ensemble_energy_max_paired_target_bh))
        to_be_added.append(normalize_target_bh_conse_unpaired(ensemble_energy_max_unpaired_target_bh))
        to_be_added.append(normalize_target_bh_overhang(ensemble_energy_5_overhang_target_bh))
        to_be_added.append(normalize_target_bh_overhang(ensemble_energy_3_overhang_target_bh))
        to_be_added.append(normalize_target_bh_paired(ensemble_energy_5_paired_target_bh))
        to_be_added.append(normalize_target_bh_paired(ensemble_energy_3_paired_target_bh))
        
        to_be_added.append(normalize_target(subopt_structures_target[0].energy))
        to_be_added.append(normalize_target_conse(ensemble_energy_max_paired_target))
        to_be_added.append(normalize_target_conse_unpaired(ensemble_energy_max_unpaired_target))
        to_be_added.append(normalize_target_overhang(ensemble_energy_5_overhang_target))
        to_be_added.append(normalize_target_overhang(ensemble_energy_3_overhang_target))
        to_be_added.append(normalize_target_paired(ensemble_energy_5_paired_target))
        to_be_added.append(normalize_target_paired(ensemble_energy_3_paired_target))
        
        to_be_added.append(normalize_seed(seed_energy))
        to_be_added.append(normalize_middle(middle_energy))
        to_be_added.append(normalize_distal(distal_energy))

        # ==========================================================
        # Add Gaussian noise independently to each feature
        # ==========================================================
        
        for feature_idx in range(num_features):
        
            if i in feature_noise_indices[feature_idx]:
        
                noise = np.random.normal(
                    loc=0.0,
                    scale=NOISE_STD
                )
        
                to_be_added[feature_idx] += noise
        
                # Keep normalized feature in [0,1]
                to_be_added[feature_idx] = np.clip(
                    to_be_added[feature_idx],
                    0.0,
                    1.0
                )
    
        energy_array_unit.append(to_be_added)
    
        energy_array_to_append = [energy_array_unit]
        energy_array.append(energy_array_to_append)

energy_array = list(itertools.chain.from_iterable(energy_array))

# Open a file in write mode
with open('Feature_MLP_0025.txt', 'a') as file:
    # Iterate over each row in the 2D array
    for row in energy_array:
        # Convert each element to a string and join them with spaces
        file.write(' '.join(map(str, row)) + '\n')
        

print('-------------')

print(min_guide_energy, max_guide_energy, min_target_bh_energy, max_target_bh_energy, min_target_energy, max_target_energy)
print(min_guide_energy_conse, max_guide_energy_conse, min_target_bh_energy_conse, max_target_bh_energy_conse, min_target_energy_conse, max_target_energy_conse)
print(min_guide_energy_conse_unpaired, max_guide_energy_conse_unpaired, min_target_bh_energy_conse_unpaired, max_target_bh_energy_conse_unpaired, min_target_energy_conse_unpaired, max_target_energy_conse_unpaired)
print(min_guide_energy_5_overhang, max_guide_energy_5_overhang, min_target_bh_energy_5_overhang, max_target_bh_energy_5_overhang, min_target_energy_5_overhang, max_target_energy_5_overhang)
print(min_guide_energy_3_overhang, max_guide_energy_3_overhang, min_target_bh_energy_3_overhang, max_target_bh_energy_3_overhang, min_target_energy_3_overhang, max_target_energy_3_overhang)
print(min_guide_energy_5_paired, max_guide_energy_5_paired, min_target_bh_energy_5_paired, max_target_bh_energy_5_paired, min_target_energy_5_paired, max_target_energy_5_paired)
print(min_guide_energy_3_paired, max_guide_energy_3_paired, min_target_bh_energy_3_paired, max_target_bh_energy_3_paired, min_target_energy_3_paired, max_target_energy_3_paired)

print(max_seed_energy, min_seed_energy)
print(max_middle_energy, min_middle_energy)
print(max_distal_energy, min_distal_energy)